# Merchant Monthly Panel — Banorte XB

**Goal**: Build `df_panel` — one row per `(merchant, year_month)` — the correct unit of analysis for churn prediction.

## Design Decisions

| Question | Decision | Rationale |
|---|---|---|
| Unit of analysis | `(merchant, year_month)` | Churn is a merchant-level event; sub-merchant rows are not independent |
| N | 20 merchants × 6 months = up to **120 rows** | Genuinely independent observation periods |
| Card type / issuer | **Features, not grouping keys** | Concentration indices capture behavioral signal without pseudo-replication |
| Fee pct | Weighted average (`sum(fee)/sum(amt_gross)`) | Simple avg gives equal weight to a $1 and a $100k txn |
| Validation (downstream) | Leave-one-merchant-out CV | Prevents leakage across merchant time series |

## 1. Setup & Data Loading

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)

DATA_DIR = Path("../data/raw/banorte_xb")

In [2]:
N_MONTHS = 6
all_files = sorted(DATA_DIR.rglob("*.parquet"))
files = all_files[-N_MONTHS:]

print(f"Total parquet files available: {len(all_files)}")
print(f"Loading last {N_MONTHS} months:")
for f in files:
    print(f"  {f.relative_to(DATA_DIR)}")

df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
print(f"\nRaw shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

Total parquet files available: 28
Loading last 6 months:
  2025\10.parquet
  2025\11.parquet
  2025\12.parquet
  2026\01.parquet
  2026\02.parquet
  2026\03.parquet

Raw shape: 7,965,406 rows x 19 columns


## 2. Type Conversion, Cleanup & Column Standardization

In [3]:
# Drop unnecessary columns
df = df.drop(columns=["PROVIDER", "PAYMENT_AMOUNT"])

# Convert date columns
df["CONFIRM_DATE"] = pd.to_datetime(df["CONFIRM_DATE"])
df["FEE_DATE"] = pd.to_datetime(df["FEE_DATE"])

# Convert numeric columns (all are strings)
numeric_cols = [
    "AMOUNT_GROSS", "NET_AMOUNT", "FEE_EBANX", "IVA_EBANX",
    "CUSTO_TRANSACAO", "AMOUNT_INSTALLMENT", "AMOUNT_IVA_INSTALLMENT",
    "INSTALLMENTS",
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Null handling: CARD_ISSUER and CARD_TYPE
df["CARD_ISSUER"] = df["CARD_ISSUER"].fillna("OTHER")
df["CARD_TYPE"] = df["CARD_TYPE"].fillna("OTHER")

print("Type conversion done.")
print(f"CUSTO_TRANSACAO nulls before imputation: {df['CUSTO_TRANSACAO'].isna().sum():,}")
print(f"CARD_ISSUER nulls:  {(df['CARD_ISSUER'] == 'OTHER').sum():,} (filled with OTHER)")
print(f"CARD_TYPE nulls:    {(df['CARD_TYPE'] == 'OTHER').sum():,} (filled with OTHER)")

Type conversion done.
CUSTO_TRANSACAO nulls before imputation: 9,117
CARD_ISSUER nulls:  21,962 (filled with OTHER)
CARD_TYPE nulls:    15 (filled with OTHER)


### CUSTO_TRANSACAO Imputation

The fee percentage (`CUSTO_TRANSACAO`) is null for some rows. Since the rate depends on the merchant's contract terms, card type, and issuer, we impute using a **cascading groupby median**:

1. **Level 1**: `(MERCHANT_NAME, FEE_DATE, CARD_TYPE, CARD_ISSUER)` — most specific
2. **Level 2**: `(MERCHANT_NAME, CARD_TYPE, CARD_ISSUER)` — same contract terms, any date
3. **Level 3**: `(MERCHANT_NAME, CARD_TYPE)` — same merchant + card type
4. **Level 4**: `(MERCHANT_NAME)` — merchant-wide fallback
5. **Level 5**: Global median — last resort

In [4]:
null_before = df["CUSTO_TRANSACAO"].isna().sum()

# Cascading group levels: most specific → least specific
group_levels = [
    ["MERCHANT_NAME", "FEE_DATE", "CARD_TYPE", "CARD_ISSUER"],
    ["MERCHANT_NAME", "CARD_TYPE", "CARD_ISSUER"],
    ["MERCHANT_NAME", "CARD_TYPE"],
    ["MERCHANT_NAME"],
]

for level_cols in group_levels:
    still_null = df["CUSTO_TRANSACAO"].isna()
    if not still_null.any():
        break
    medians = df.groupby(level_cols)["CUSTO_TRANSACAO"].transform("median")
    df.loc[still_null, "CUSTO_TRANSACAO"] = medians[still_null]
    filled = null_before - df["CUSTO_TRANSACAO"].isna().sum()
    print(f"Level {group_levels.index(level_cols) + 1} {level_cols!s:55s} → filled {filled:,} total so far")
    null_before = df["CUSTO_TRANSACAO"].isna().sum()

# Global fallback
still_null = df["CUSTO_TRANSACAO"].isna()
if still_null.any():
    global_median = df["CUSTO_TRANSACAO"].median()
    df.loc[still_null, "CUSTO_TRANSACAO"] = global_median
    print(f"Level 5 global median ({global_median:.6f})          → filled {still_null.sum():,}")

print(f"\nCUSTO_TRANSACAO nulls after imputation: {df['CUSTO_TRANSACAO'].isna().sum()}")

Level 1 ['MERCHANT_NAME', 'FEE_DATE', 'CARD_TYPE', 'CARD_ISSUER'] → filled 9,108 total so far
Level 2 ['MERCHANT_NAME', 'CARD_TYPE', 'CARD_ISSUER']           → filled 9 total so far

CUSTO_TRANSACAO nulls after imputation: 0


In [5]:
# Quick check: distribution of imputed fee_pct should be reasonable
print("CUSTO_TRANSACAO distribution after imputation:")
print(df["CUSTO_TRANSACAO"].describe())
print("Any remaining nulls across all columns:")
remaining = df.isna().sum()
remaining = remaining[remaining > 0]
print(remaining if len(remaining) > 0 else "  None")

CUSTO_TRANSACAO distribution after imputation:
count   7,965,406.0000
mean            0.0163
std             0.0014
min             0.0145
25%             0.0145
50%             0.0170
75%             0.0170
max             0.0320
Name: CUSTO_TRANSACAO, dtype: float64
Any remaining nulls across all columns:
  None


In [6]:
# Rename: english, lowercase, abbreviated, no "ebanx" branding
df = df.rename(columns={
    "MERCHANT_NAME": "merchant",
    "CONFIRM_DATE": "confirm_date",
    "FEE_DATE": "fee_date",
    "AMOUNT_GROSS": "amt_gross",
    "NET_AMOUNT": "amt_net",
    "FEE_EBANX": "fee",
    "IVA_EBANX": "iva",
    "CUSTO_TRANSACAO": "fee_pct",
    "AMOUNT_INSTALLMENT": "amt_installment",
    "AMOUNT_IVA_INSTALLMENT": "amt_iva_installment",
    "INSTALLMENTS": "installments",
    "CARD_TYPE": "card_type",
    "CARD_ISSUER": "card_issuer",
    "CARD_SCHEME": "card_scheme",
    "OPERATION_TYPE": "op_type",
    "CAMARA_COMPENSACION": "clearing_house",
    "AFILIACION": "affiliation",
})

print(f"Columns ({len(df.columns)}):")
print(df.columns.tolist())

Columns (17):
['card_type', 'fee_date', 'card_issuer', 'op_type', 'amt_gross', 'amt_installment', 'installments', 'merchant', 'card_scheme', 'confirm_date', 'clearing_house', 'fee_pct', 'fee', 'amt_iva_installment', 'amt_net', 'iva', 'affiliation']


## 3. Monthly Panel Aggregation

Group by `(merchant, year_month)` — one row per merchant per calendar month.
Same metrics as before, now with a time dimension: **20 merchants × 6 months = up to 120 rows**.

**Percentage calculations** (weighted averages):
- `avg_fee_pct` = sum(fee) / sum(amt_gross)
- `avg_iva_pct` = sum(iva) / sum(fee)
- `avg_installment_pct` = sum(amt_installment) / sum(amt_gross) — installment txns only
- `avg_iva_installment_pct` = sum(amt_iva_installment) / sum(amt_installment) — installment txns only

In [7]:
df["year_month"] = df["confirm_date"].dt.to_period("M")

months = sorted(df["year_month"].unique())
print(f"Months in dataset ({len(months)}): {[str(m) for m in months]}")
print(f"Total rows: {len(df):,}")

Months in dataset (6): ['2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03']
Total rows: 7,965,406


In [8]:
PANEL_COLS = ["merchant", "year_month"]

g = df.groupby(PANEL_COLS)
df_inst = df[df["installments"] > 1]
g_inst = df_inst.groupby(PANEL_COLS)

# Base aggregations
df_panel = g.agg(
    total_txs=("amt_gross", "size"),
    avg_ticket=("amt_gross", "mean"),
    median_ticket=("amt_gross", "median"),
    total_amt_gross=("amt_gross", "sum"),
    total_amt_net=("amt_net", "sum"),
    total_fee=("fee", "sum"),
    total_iva=("iva", "sum"),
    total_amt_installment=("amt_installment", "sum"),
    total_amt_iva_installment=("amt_iva_installment", "sum"),
)

# Installment tx count & ratio
installment_txs = g_inst.size().rename("installment_txs")
df_panel = df_panel.join(installment_txs, how="left")
df_panel["installment_txs"] = df_panel["installment_txs"].fillna(0).astype(int)
df_panel["installment_ratio"] = df_panel["installment_txs"] / df_panel["total_txs"]

# Weighted average percentages
df_panel["avg_fee_pct"] = df_panel["total_fee"] / df_panel["total_amt_gross"]
df_panel["avg_iva_pct"] = np.where(df_panel["total_fee"] > 0, df_panel["total_iva"] / df_panel["total_fee"], 0)

# Installment percentages (installment txns only)
inst_gross = g_inst["amt_gross"].sum().rename("_inst_gross")
inst_amt   = g_inst["amt_installment"].sum().rename("_inst_amt")
inst_iva   = g_inst["amt_iva_installment"].sum().rename("_inst_iva")
df_panel = df_panel.join(inst_gross).join(inst_amt).join(inst_iva)
df_panel["avg_installment_pct"]     = np.where(df_panel["_inst_gross"] > 0, df_panel["_inst_amt"] / df_panel["_inst_gross"], 0)
df_panel["avg_iva_installment_pct"] = np.where(df_panel["_inst_amt"]   > 0, df_panel["_inst_iva"] / df_panel["_inst_amt"],   0)
df_panel = df_panel.drop(columns=["_inst_gross", "_inst_amt", "_inst_iva"])

df_panel = df_panel.reset_index().sort_values(PANEL_COLS).reset_index(drop=True)

print(f"df_panel shape: {df_panel.shape[0]:,} merchant-months x {df_panel.shape[1]} columns")
print(f"Unique merchants: {df_panel['merchant'].nunique()}")
print(f"Unique months:    {df_panel['year_month'].nunique()}")

df_panel shape: 95 merchant-months x 17 columns
Unique merchants: 20
Unique months:    6


## 4. Verification & Preview

In [9]:
df_panel.head(12)

,merchant,year_month,total_txs,avg_ticket,median_ticket,total_amt_gross,total_amt_net,total_fee,total_iva,total_amt_installment,total_amt_iva_installment,installment_txs,installment_ratio,avg_fee_pct,avg_iva_pct,avg_installment_pct,avg_iva_installment_pct
0,Adobe - Braintree,2025-10,42635,481.7104,332.4900,"20,537,721.8700","20,135,918.5803","346,382.1463","55,421.1434",0.0000,0.0000,0,0.0000,0.0169,0.1600,0.0000,0.0000
1,Adobe - Braintree,2025-11,50464,486.4851,332.4900,"24,549,983.2300","24,069,254.8377","414,421.0279","66,307.3645",0.0000,0.0000,0,0.0000,0.0169,0.1600,0.0000,0.0000
2,Adobe - Braintree,2025-12,60548,489.4375,332.4900,"29,634,463.8800","29,054,465.9447","499,998.2201","79,999.7152",0.0000,0.0000,0,0.0000,0.0169,0.1600,0.0000,0.0000
3,Adobe - Braintree,2026-01,61274,508.3061,375.7300,"31,145,944.9900","30,537,290.9747","524,701.7373","83,952.2780",0.0000,0.0000,0,0.0000,0.0168,0.1600,0.0000,0.0000
4,Adobe - Braintree,2026-02,53996,501.9663,388.0900,"27,104,174.3600","26,573,561.7854","457,424.6333","73,187.9413",0.0000,0.0000,0,0.0000,0.0169,0.1600,0.0000,0.0000
5,Alibaba ICBU,2025-10,6316,"13,519.1622","5,475.0950","85,387,028.4600","83,706,344.9271","1,448,167.0443","231,706.7271",698.0702,111.6912,1,0.0002,0.0170,0.1600,0.0920,0.1600
6,Alibaba ICBU,2025-11,6311,"12,826.9310","4,925.0700","80,950,761.5000","79,357,441.9572","1,373,551.3300","219,768.2128",0.0000,0.0000,0,0.0000,0.0170,0.1600,0.0000,0.0000
7,Alibaba ICBU,2025-12,7073,"13,327.7973","5,106.0700","94,267,510.3500","92,372,552.4112","1,602,560.9346","256,409.7495","31,023.4954","4,963.7593",64,0.0090,0.0170,0.1600,0.0222,0.1600
8,Alibaba ICBU,2026-01,14643,"12,600.8792","4,981.2800","184,514,674.2600","180,386,351.3519","3,126,958.9386","500,313.4302","431,940.1201","69,110.4192",1538,0.1050,0.0169,0.1600,0.0181,0.1600
9,Alibaba ICBU,2026-02,8483,"10,304.2193","3,359.4200","87,410,692.2500","85,421,928.5974","1,482,541.1386","237,206.5822","231,910.2861","37,105.6458",963,0.1135,0.0170,0.1600,0.0196,0.1600


In [10]:
df_panel.describe()

,total_txs,avg_ticket,median_ticket,total_amt_gross,total_amt_net,total_fee,total_iva,total_amt_installment,total_amt_iva_installment,installment_txs,installment_ratio,avg_fee_pct,avg_iva_pct,avg_installment_pct,avg_iva_installment_pct
count,95.0000,95.0000,95.0000,95.0000,95.0000,95.0000,95.0000,95.0000,95.0000,95.0000,95.0000,95.0000,95.0000,95.0000,95.0000
mean,"83,846.3789","1,380.1405",680.2364,"41,479,069.3793","40,674,302.2969","675,149.2875","108,023.8860","18,615.4387","2,978.4702",468.6421,0.0160,0.0170,0.1600,0.0048,0.0337
std,"203,903.5845","2,827.0521","1,171.9106","94,039,541.4976","92,292,841.0539","1,489,711.8310","238,353.8930","65,964.1685","10,554.2670","1,519.2939",0.0539,0.0015,0.0000,0.0130,0.0656
min,1.0000,6.0000,7.5000,96.0000,92.7149,2.8320,0.4531,0.0000,0.0000,0.0000,0.0000,0.0147,0.1600,0.0000,0.0000
25%,695.0000,155.0241,126.7500,"850,311.3400","834,088.9294","14,450.3734","2,312.0597",0.0000,0.0000,0.0000,0.0000,0.0168,0.1600,0.0000,0.0000
50%,"14,606.0000",443.7652,259.0000,"4,367,138.1900","4,275,816.8770","77,695.4098","12,431.2656",0.0000,0.0000,0.0000,0.0000,0.0169,0.1600,0.0000,0.0000
75%,"53,811.5000","1,146.1516",595.7500,"26,934,979.2150","26,381,874.4413","455,597.5715","72,895.6114",0.0000,0.0000,0.0000,0.0000,0.0170,0.1600,0.0000,0.0000
max,"1,097,876.0000","13,519.1622","5,475.0950","496,046,011.1400","487,497,823.0265","7,274,445.3156","1,163,911.2505","431,940.1201","69,110.4192","9,581.0000",0.4167,0.0295,0.1600,0.0920,0.1600


In [11]:
# Sanity check: weighted avg_fee_pct matches manual calculation for top merchant-month
top_row = df_panel.sort_values("total_txs", ascending=False).iloc[0]
label = f"{top_row['merchant']} — {top_row['year_month']}"

print(f"Sanity check for: {label}")
print(f"  total_fee / total_amt_gross = {top_row['total_fee'] / top_row['total_amt_gross']:.6f}")
print(f"  avg_fee_pct (stored)        = {top_row['avg_fee_pct']:.6f}")
print(f"  Match: {np.isclose(top_row['total_fee'] / top_row['total_amt_gross'], top_row['avg_fee_pct'])}")
print()
print(f"  total_iva / total_fee       = {top_row['total_iva'] / top_row['total_fee']:.6f}")
print(f"  avg_iva_pct (stored)        = {top_row['avg_iva_pct']:.6f}")
print(f"  Match: {np.isclose(top_row['total_iva'] / top_row['total_fee'], top_row['avg_iva_pct'])}")

Sanity check for: Temu.com — 2025-12
  total_fee / total_amt_gross = 0.014665
  avg_fee_pct (stored)        = 0.014665
  Match: True

  total_iva / total_fee       = 0.160000
  avg_iva_pct (stored)        = 0.160000
  Match: True


In [12]:
print(f"Final df_panel columns ({len(df_panel.columns)}):")
print(df_panel.columns.tolist())
print(f"\nShape: {df_panel.shape[0]:,} merchant-months x {df_panel.shape[1]} columns")
print(f"Nulls: {df_panel.isna().sum().sum()}")

Final df_panel columns (17):
['merchant', 'year_month', 'total_txs', 'avg_ticket', 'median_ticket', 'total_amt_gross', 'total_amt_net', 'total_fee', 'total_iva', 'total_amt_installment', 'total_amt_iva_installment', 'installment_txs', 'installment_ratio', 'avg_fee_pct', 'avg_iva_pct', 'avg_installment_pct', 'avg_iva_installment_pct']

Shape: 95 merchant-months x 17 columns
Nulls: 0
